# Day 6: LLM Architecture

last day of week 4. day 3 covered encoder based transformers like BERT, which see the whole input at once and are good at understanding text. this one's about decoder only models, GPT style, which generate one word at a time, left to right, where each new word can only see the words that came before it since the rest hasn't been generated yet. that's why GPT style models are good at generating text and BERT style models are good at understanding it.

using plain GPT-2 here, no instruction tuning, no RLHF, just the base pretrained model. that's on purpose, it makes the comparison in this notebook actually show something real instead of a vague explanation.

In [1]:
import ssl
import certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

num_params = sum(p.numel() for p in model.parameters())
print("GPT-2 loaded.")
print("Number of parameters:", num_params)
print("Context window (max tokens):", model.config.n_ctx)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9732.33it/s]

GPT-2 loaded.
Number of parameters: 124439808
Context window (max tokens): 1024


two concepts with real numbers attached now instead of just definitions. parameters are the actual learned numbers inside the model, 124.4 million of them here, this is a genuinely small model by today's standards, modern LLMs run into the billions. context window is the max number of tokens the model can look at once, input and output combined, 1024 here. anything past that limit gets cut off or forgotten.

In [2]:
prompt = "The stock market today"

inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=30, do_sample=False) ##do_sample=False is an example of 'greedy' decoding in which it always picks the most likely next token, keep this reproducible
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("Prompt:", prompt)
print("Generated:", generated_text)
## this has produced a 'stuck' output in which the GPT-2 model is repeating itself after a few unique tokens
##limitation of the model essentially, has no instruction tuning or reasoning to the model

Prompt: The stock market today
Generated: The stock market today is a very volatile market. The market is volatile because of the volatility of the stock market. The stock market is volatile because of the volatility of the


this is decoder only generation actually happening. tokenize the prompt, then the model repeatedly predicts the next token, appends it, and repeats. each new word only sees the words before it, never what comes after, since it hasn't been generated yet. total opposite of day 3's encoder which sees the whole input at once.

the output gets stuck repeating itself, volatile market, volatility of the stock market, volatile because of the volatility, over and over. that's a real known failure mode of greedy decoding specifically, always picking the single most likely next word tends to create loops since once a pattern starts, the most likely next word just keeps reinforcing that same pattern. it's also why real products don't use plain greedy decoding, they use smarter sampling methods like temperature or top-k/top-p to avoid exactly this.

In [3]:
##compare this output in regard to an instruction style prompt to see if the phrasing wil change the output at all
instruction_prompt = "Question: What happened in the stock market today?\nAnswer:"

inputs2 = tokenizer(instruction_prompt, return_tensors="pt")
output2 = model.generate(**inputs2, max_new_tokens=30, do_sample=False)
generated_text2 = tokenizer.decode(output2[0], skip_special_tokens=True)
print("\nPrompt 2:", instruction_prompt)
print("Generated 2:", generated_text2)


Prompt 2: Question: What happened in the stock market today?
Answer:
Generated 2: Question: What happened in the stock market today?
Answer: The stock market is a very volatile market. It is not a perfect market. It is not a perfect market. It is not a perfect market.


this is the actual practical task, comparing outputs from different prompts. formatted this one as a proper question and answer style prompt to see if that phrasing gets a real answer out of it.

it didn't. still loops, this time on "it is not a perfect market" three times in a row, instead of actually answering the question. that's the real point here, phrasing something as a question doesn't automatically make a base model understand it should answer helpfully. there's no instruction following behavior built in, it's still just predicting likely next words based on patterns in its training text, question and answer format included, and it happened to fall into a loop either way.

## pretraining, instruction tuning, and RLHF

what we just saw is exactly why modern LLMs go through more than one training stage.

pretraining is the first and only stage this GPT-2 model went through. predict the next word on huge amounts of raw internet text, over and over. that gets you a base model, it knows language and picks up on real patterns, but it has no concept of being helpful or following instructions, which is exactly what we saw above.

instruction tuning is the next stage. take a base model and fine-tune it specifically on examples of instructions paired with good responses, so it learns the pattern of actually answering instead of just continuing text in whatever direction feels statistically likely.

RLHF, reinforcement learning from human feedback, goes a step further. humans rank multiple model responses against each other, that ranking data trains a separate reward model, and then the LLM gets fine-tuned again using reinforcement learning to push its outputs toward whatever the reward model says humans prefer.

not building RLHF here, and that's intentional, not a shortcut. real RLHF needs a trained reward model, large scale human preference data, and reinforcement learning algorithms to fine-tune against that reward model, genuinely heavy infrastructure that isn't something you'd run locally. even at real companies it's usually a specialized team's job, not something every engineer implements from scratch. the curriculum lists it as an overview for a reason, understanding why it exists matters more here than actually running it.

## base vs instruct vs chat models

base model: just continues text, exactly what we tested above. no instruction following, no conversation awareness.

instruct model: went through instruction tuning on top of the base model, tuned to follow a single instruction well, give it a task and it tries to actually do that task instead of just continuing the sentence.

chat model: tuned specifically for multi-turn conversation, remembers context across a back and forth exchange, this is what things like ChatGPT actually are under the hood, base model, then instruction tuning, then RLHF, then further tuned for conversation specifically.

## takeaway

this closes out week 4. same general lesson keeps showing up across this whole project in different forms, a pretrained model by itself is only ever a starting point, not a finished product. day 4 needed fine-tuning to actually classify our headlines well. this GPT-2 base model needs instruction tuning and RLHF on top of pretraining before it can act like an actual assistant instead of just continuing text patterns, loops included.

week 5 is prompt engineering, running a local or API LLM, fine-tuning concepts, and building an actual RAG chatbot, which is really where a lot of this project comes together, our own scraped and cleaned news data could genuinely become the knowledge base a RAG system retrieves from.